# Figure S1 - raw heatmap

In [ ]:
from datetime import datetime
import pandas as pd
import numpy as np
import os
import time

import plotly.offline as pyo
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio

pio.templates.default = "plotly_white" # https://medium.com/plotly/introducing-plotly-py-theming-b644109ac9c7

directory = r'C:\Users\markm\OneDrive\Akadeemiline\PhD RESEARCH\Artiklid\Ukraine-Twitter\Data\Final datasets used in article'
os.chdir(directory)

remove_UA_RU=False
smooth=True
smooth_time=6
Filter=False


In [ ]:
DFfreq = pd.read_csv(r'pivotUkraine_SUM_excelInput_weekly.csv') #, index_col=0, sep=';'
#Load frequency share pivoted df
DFfreq.set_index(DFfreq.columns[0], inplace=True)
DFfreq.head(3)

In [ ]:
if remove_UA_RU:
    DFfreq = DFfreq.drop(index=['Ukrainian', 'Russian'])

manual_order = [
     'Ukrainian', 'Russian', 'Arabic', 'Portuguese', 'Catalan', 'Korean',
     'Persian', 'Turkish', 'Indonesian', 'Urdu', 'Vietnamese',
     'Serbian', 'Estonian', 'Romanian', 'Greek', 'Hungarian', 'Polish',
     'Swedish', 'Czech', 'Spanish', 'Danish', 'English', 'Dutch',
     'Norwegian', 'Finnish', 'French', 'Italian', 'German'
]

#manual_order.reverse()

# FIX: reorder index instead of using a missing column
DFfreq = DFfreq.reindex(manual_order)

# Apply log transform
DFfreqLog = DFfreq.applymap(
    lambda x: 0 if x == 0 else (np.log10(x * 100000) if x > 0 else -np.log10(-x * 100000))
)

DFfreqLog = DFfreq.applymap(lambda x: np.nan if x==0 else np.log10(x*100000))


In [ ]:
DFfreqLog

# Visualization

In [ ]:
#Unpivoting table from wide to long format
AttentionLog=DFfreqLog.reset_index().melt(id_vars=['language'], var_name='Date', value_name='ExpressionLog')
AttentionNonLog=DFfreq.reset_index().melt(id_vars=['language'], var_name='Date', value_name='Expression')
#Freq=DFfreq.reset_index().melt(id_vars=['language'], var_name='Date', value_name='FractionRaw')

# Merge freq and attention share dfs
merged_df = Freq.merge(AttentionLog, on=['language', 'Date'], how='inner')
merged_df = merged_df.merge(AttentionNonLog, on=['language', 'Date'], how='inner')
#display(merged_df)
#print(merged_df.describe())
merged_df['FractionLog'] = np.log10(merged_df['FractionRaw'])+7.0 #Cant do size with neg values

merged_df['Fraction']= merged_df['FractionRaw'].replace(0, np.nan)
merged_df['FractionLog']=np.log10(merged_df.Fraction)
#print(merged_df.describe())

merged_df['FractionLog']= (merged_df['FractionLog'])+7.5 #Cant do size with neg values
merged_df['FractionLog']=merged_df['FractionLog'].fillna(0)
merged_df['FractionRaw']=merged_df['FractionRaw'].fillna(0)

manual_order = [
     'Ukrainian', 'Russian', 'Arabic', 'Portuguese', 'Catalan', 'Korean', 'Persian', 'Turkish', 'Indonesian', 'Urdu', 'Vietnamese',
    'Serbian', 'Estonian', 'Romanian', 'Greek', 'Hungarian', 'Polish', 'Swedish', 'Czech', 'Spanish', 
    'Danish', 'English', 'Dutch', 'Norwegian','Finnish', 'French', 'Italian', 'German'
]
'''
#NEW ORDER
manual_order = [
    'Ukrainian', 'Russian', 'Urdu', 'Vietnamese', 'Serbian', 'Estonian', 'Hungarian',
    'Turkish', 'Arabic', 'Korean', 'Catalan', 'Indonesian', 'Portuguese',
    'Polish', 'Czech', 'Norwegian', 'Finnish', 'Dutch', 'English', 'Swedish', 'Danish',
    'French', 'Italian', 'German', 'Spanish', 'Greek', 'Romanian'
]
'''

manual_order.reverse()

merged_df['language'] = pd.Categorical(merged_df['language'], categories=manual_order, ordered=True)
merged_df = merged_df.sort_values(['language', 'Date'])


In [ ]:
import plotly.graph_objects as go
import numpy as np

#For color scale bar
min_val = DFfreqLog.values.min()
max_val = DFfreqLog.values.max()

custom_colorscale = [
    [0, "#eeeeee"],  # very light gray (min non-zero)
    [1, "#000000"]   # black (max)
]

# choose ticks similar to your example (edit as needed)
tickvals = np.linspace(min_val, max_val, 7)

# tick text
#ticktext = [f"{v:.2f}" for v in tickvals] #Pure log values
#ticktext = [f"{0 if v==0 else 10**v / 100000:.4f}" for v in tickvals] #nonlog static freq values
#ticktext = [f"{0 if v==0 else 10**v / 100000 * 100:.5f}%" for v in tickvals] #nonlogstatic percentages

#'''
#  nonlog dynamic percentages
# convert log positions back to linear percentages
# convert log positions back to linear percentages
# convert log positions back to linear percentages
linear_values = [0 if v==0 else 10**v / 100000 * 100 for v in tickvals]

def format_percentage_dynamic(val):
    if val == 0:
        return "0%"
    # For very small numbers, use scientific notation
    if val < 0.01:   # adjust threshold as needed
        return f"{val:.2g}%"
    else:
        # Otherwise, normal decimal with up to 2 significant digits after first non-zero
        return f"{val:.2f}%"

ticktext = [format_percentage_dynamic(v) for v in linear_values]

#'''


fig = go.Figure()

fig.add_trace(
    go.Heatmap(
        z=DFfreqLog.values,
        x=DFfreqLog.columns,
        y=DFfreqLog.index,
        colorscale="Greys",
        zmin=min_val,
        zmax=max_val,

        colorbar=dict(
            orientation="h",
            y=-0.18,
            x=0.5,
            xanchor="center",

            # LENGTH & THICKNESS
            len=0.50,               # long bar like in your sample
            thickness=45,           # thick boxed bar
            thicknessmode="pixels",

            # FULL BLACK BORDER AROUND BAR (KEY PART)
            outlinewidth=2,
            outlinecolor="black",

            # TICKS (in your example they were outside the box)
            tickvals=tickvals,
            ticktext=ticktext,
            ticks="outside",
            ticklen=10,
            tickcolor="black",

            # POSITION OF TICK LABELS – adjust depending on your example
            ticklabelposition="outside bottom",

            # FONT
            tickfont=dict(size=27, color="black"),
        )
    )
)

fig.update_layout(
    height=1400,
    width=2100,
    margin=dict(l=70, r=40, t=60, b=200),

    # X-axis (dates) font
    xaxis=dict(
        title=" ",
        titlefont=dict(size=34, color="black"),  # x-axis title
        tickfont=dict(size=34, color="black")    # x-axis tick labels
    ),

    # Y-axis (languages) font
    yaxis=dict(
        title=" ",
        #titlefont=dict(size=18, color="black"),  # y-axis title
        tickfont=dict(size=32, color="black"),   # y-axis tick labels
        autorange="reversed"
    ),

    # General title
    title=dict(
        text="",
        font=dict(size=30, color="black")
    )
)

fig.show()


In [ ]:
import numpy as np
import plotly.graph_objects as go

# Replace 0 with NaN for plotting (white cells)
DFfreqLog_plot = DFfreq.applymap(lambda x: np.nan if x==0 else np.log10(x*100000))

# Get min/max of non-NaN values
min_val = np.nanmin(DFfreqLog_plot.values)
max_val = np.nanmax(DFfreqLog_plot.values)
# DFfreq has the original non-log values
nonzero_vals = DFfreq.values.flatten()
nonzero_vals = nonzero_vals[nonzero_vals > 0]  # ignore zeros
min_percent = nonzero_vals.min() * 100
max_percent = nonzero_vals.max() * 100

custom_colorscale = [
    [0, "#f0f0f0"],  # very light gray (min non-zero)
    [1, "#000000"]   # black (max)
]

# Define tick positions manually: include 0 + log positions of non-zero data
tickvals = np.linspace(min_val, max_val, 6)  # log positions
tickvals = np.insert(tickvals, 0, 0)  # explicitly add zero at start

# Convert tick positions back to percentages
# Desired tick percentages
linear_tick_percentages = [0.0001, 0.001, 0.01, 0.1, 1, min_percent, max_percent]
linear_tick_percentages = sorted(set(linear_tick_percentages))  # sort & remove duplicates

def percent_to_log(val):
    if val == 0:
        return -1.5  # placeholder for zero
    return np.log10(val / 100 * 100000)

tickvals = [percent_to_log(v) for v in linear_tick_percentages]
ticktext = [f"{v:.4g}%" for v in linear_tick_percentages]  # dynamic formatting

fig = go.Figure()

fig.add_trace(
    go.Heatmap(
        z=DFfreqLog_plot.values,
        x=DFfreqLog_plot.columns,
        y=DFfreqLog_plot.index,
        colorscale=custom_colorscale,
        zmin=min_val,
        zmax=max_val,
        colorbar=dict(
            orientation="h",
            y=-0.25,
            x=0.5,
            xanchor="center",
            len=0.50,
            thickness=45,
            thicknessmode="pixels",
            outlinewidth=2,
            outlinecolor="black",
            ticks="outside",
            tickvals=tickvals,
            ticktext=ticktext,
            ticklabelposition="outside bottom",
            ticklen=10,
            tickcolor="black",
            tickfont=dict(size=29, color="black")
        )
    )
)

fig.update_layout(
    height=1400,
    width=2100,
    margin=dict(l=70, r=40, t=60, b=200),
    xaxis=dict(title=" ", tickfont=dict(size=34, color="black")),
    yaxis=dict(title=" ", tickfont=dict(size=31, color="black"), autorange="reversed"),
    title=dict(text="", font=dict(size=30, color="black"))
)

fig.show()


In [ ]:
import os
from subprocess import call

# Set the figure directory and filename criteria
sublocation = f'/Visualizations/Fig.S1_raw freq heatmap/'
# Define the plot name
plotname = 'rawfreqplot'  # Replace with your desired plot name

# Choose the name based on criteria
if remove_UA_RU:
    name = f'{plotname}_Smooth_NO-UA-RU' if smooth else f'{plotname}_NO-Smooth_NO-UA-RU'
else:
    name = f'{plotname}_Smooth_with-UA-RU' if smooth else f'{plotname}_NO-Smooth_with-UA-RU'

# Define the base directory for saving files
directory = r'C:\Users\markm\OneDrive\Akadeemiline\PhD RESEARCH\Artiklid\Ukraine-Twitter'
os.chdir(directory)

# Make the main figure directory if not present
main_figure_path = directory + sublocation
os.makedirs(main_figure_path, exist_ok=True)

# Define output formats and create their subdirectories
formats = ['pdf', 'jpeg', 'svg']
for fmt in formats:
    os.makedirs(main_figure_path + fmt + "/", exist_ok=True)  # Use forward slash for consistency

# Save the figure in different formats
for fmt in formats:
    file_path = main_figure_path + fmt + "/" + "_" + name + "." + fmt  # Ensure consistent path formatting
    fig.write_image(file_path, format=fmt, engine="kaleido")
    print(f"Saved {fmt.upper()} to: {file_path}")

# Convert PDF to EPS if needed
# call(["pdf2ps", main_figure_path + "pdf" + "/" + "_" + name + ".pdf", main_figure_path + "eps" + "/" + "_" + name + ".eps"])
